In [6]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Enable proxy configuration for Codio/JupyterLab environment
JupyterDash.infer_jupyter_proxy_config()

# Import the CRUD module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Connect to the AAC database via the CRUD module
username = "aacuser"
password = "SNHU1234"
db = AnimalShelter(username, password)

# Retrieve all records for the initial unfiltered table view.
# The read() method accepts an empty dict to return all documents.
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ returns an '_id' field with type ObjectID which crashes DataTable.
# Drop it here; the CRUD read() method already excludes it via projection,
# but this guard catches any edge cases.
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Load and encode the Grazioso Salvare logo for display
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([

    # -----------------------------------------------------------------------
    # Header: Logo + Title + Unique Identifier
    # -----------------------------------------------------------------------
    html.Div([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '120px', 'float': 'left', 'margin': '10px'}
        ),
        html.Div([
            html.Center(html.B(html.H1(
                'Grazioso Salvare Animal Rescue Dashboard',
                style={'color': '#2c3e50'}
            ))),
            html.Center(html.H5(
                'Kyle Dauk | CS-340 Client/Server Development | SNHU',
                style={'color': '#7f8c8d', 'font-style': 'italic'}
            )),
        ], style={'overflow': 'hidden', 'padding': '20px 10px'})
    ], style={'overflow': 'hidden', 'background-color': '#f8f9fa',
              'border-bottom': '2px solid #dee2e6', 'margin-bottom': '10px'}),

    html.Hr(),

    # -----------------------------------------------------------------------
    # Interactive Filter: Radio Items for Rescue Type Selection
    # -----------------------------------------------------------------------
    html.Div([
        html.P('Select Rescue Type to Filter Dogs:', style={
            'font-weight': 'bold', 'font-size': '16px', 'margin-bottom': '8px'
        }),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': '  Water Rescue', 'value': 'Water Rescue'},
                {'label': '  Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness Rescue'},
                {'label': '  Disaster or Individual Tracking', 'value': 'Disaster or Individual Tracking'},
                {'label': '  Reset (Show All Animals)', 'value': 'Reset'},
            ],
            value='Reset',   # Default: unfiltered view on load
            labelStyle={
                'display': 'inline-block',
                'margin-right': '30px',
                'font-size': '14px'
            }
        )
    ], style={'padding': '15px 20px', 'background-color': '#eaf2fb',
              'border-radius': '6px', 'margin': '10px 0'}),

    html.Hr(),

    # -----------------------------------------------------------------------
    # Interactive Data Table
    # -----------------------------------------------------------------------
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict('records'),

        # Table interaction settings
        editable=False,
        filter_action="native",    # Enables per-column text filtering
        sort_action="native",      # Enables column header sorting
        sort_mode="multi",         # Allow sorting by multiple columns
        column_selectable=False,
        row_selectable="single",   # Single-row selection drives the map marker
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],         # Pre-select row 0 so map renders on load

        # Pagination
        page_action="native",
        page_current=0,
        page_size=10,

        # Visual styling
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'padding': '6px 10px',
            'maxWidth': '160px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis',
            'font-size': '13px'
        },
        style_header={
            'backgroundColor': '#2c3e50',
            'color': 'white',
            'fontWeight': 'bold',
            'font-size': '13px'
        },
        style_data_conditional=[
            {'if': {'row_index': 'odd'}, 'backgroundColor': '#f9f9f9'}
        ],
    ),

    html.Br(),
    html.Hr(),

    # -----------------------------------------------------------------------
    # Visualization Row: Pie Chart (left) + Geolocation Map (right)
    # -----------------------------------------------------------------------
    html.Div(
        className='row',
        style={'display': 'flex', 'gap': '20px', 'padding': '0 10px'},
        children=[
            html.Div(id='graph-id', className='col s12 m6',
                     style={'flex': '1', 'min-width': '300px'}),
            html.Div(id='map-id', className='col s12 m6',
                     style={'flex': '1', 'min-width': '400px'})
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

# ---------------------------------------------------------------------------
# Callback 1: Update data table when rescue-type filter changes
# ---------------------------------------------------------------------------
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    """
    Query MongoDB with the appropriate filter for the selected rescue type
    and return the result as a list of record dicts for the DataTable.
    """

    # ------------------------------------------------------------------ #
    # Water Rescue — Labrador Retriever Mix, Chesapeake Bay Retriever,    #
    # Newfoundland, Portuguese Water Dog                                  #
    # Sex: Intact Female | Age: 26–156 weeks                              #
    # ------------------------------------------------------------------ #
    if filter_type == 'Water Rescue':
        df_filtered = pd.DataFrame.from_records(db.read({
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland",
                    "Portuguese Water Dog"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }))

    # ------------------------------------------------------------------ #
    # Mountain or Wilderness Rescue — German Shepherd, Alaskan Malamute, #
    # Old English Sheepdog, Siberian Husky, Rottweiler                   #
    # Sex: Intact Male | Age: 26–156 weeks                               #
    # ------------------------------------------------------------------ #
    elif filter_type == 'Mountain or Wilderness Rescue':
        df_filtered = pd.DataFrame.from_records(db.read({
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }))

    # ------------------------------------------------------------------ #
    # Disaster or Individual Tracking — Doberman Pinsch, German Shepherd, #
    # Golden Retriever, Bloodhound, Rottweiler                            #
    # Sex: Intact Male | Age: 20–300 weeks                               #
    # ------------------------------------------------------------------ #
    elif filter_type == 'Disaster or Individual Tracking':
        df_filtered = pd.DataFrame.from_records(db.read({
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinsch",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20.0, "$lte": 300.0}
        }))

    # ------------------------------------------------------------------ #
    # Reset — return all animals, no filter applied                       #
    # ------------------------------------------------------------------ #
    else:
        df_filtered = pd.DataFrame.from_records(db.read({}))

    # Clean up any _id field that may come through
    if '_id' in df_filtered.columns:
        df_filtered.drop(columns=['_id'], inplace=True)

    return df_filtered.to_dict('records')


# ---------------------------------------------------------------------------
# Callback 2: Update breed pie chart from current table data
# ---------------------------------------------------------------------------
@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    """
    Build a pie chart showing the top 10 dog breeds in the currently filtered
    data set. Driven by derived_virtual_data so it reflects sorting/filtering.
    """
    if viewData is None or len(viewData) == 0:
        return [html.P("No data to display. Adjust filters.",
                       style={'color': 'grey', 'padding': '20px'})]

    dff = pd.DataFrame.from_dict(viewData)

    if 'breed' not in dff.columns or dff.empty:
        return [html.P("Breed data unavailable.",
                       style={'color': 'grey', 'padding': '20px'})]

    # Count breed occurrences; take top 10 for readability
    breed_counts = (
        dff['breed']
        .value_counts()
        .head(10)
        .reset_index()
    )
    breed_counts.columns = ['breed', 'count']

    fig = px.pie(
        breed_counts,
        values='count',
        names='breed',
        title='Preferred Dog Breeds by Rescue Category',
        color_discrete_sequence=px.colors.sequential.RdBu
    )
    fig.update_traces(textposition='inside', textinfo='percent+label')
    fig.update_layout(showlegend=True, title_x=0.5)

    return [dcc.Graph(figure=fig)]


# ---------------------------------------------------------------------------
# Callback 3: Highlight selected columns in the data table
# ---------------------------------------------------------------------------
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    """Apply a highlight color to any selected column."""
    return [
        {'if': {'column_id': i}, 'background_color': '#D2F3FF'}
        for i in selected_columns
    ]


# ---------------------------------------------------------------------------
# Callback 4: Update geolocation map for the selected table row
# ---------------------------------------------------------------------------
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    """
    Render a Leaflet map centered on the animal selected in the data table.
    Shows a marker with the animal's breed as a tooltip and name in a popup.
    Defaults to Austin, TX (30.75, -97.48) if no row is selected.
    """
    if viewData is None or len(viewData) == 0:
        # Default center — Austin, TX
        return [
            dl.Map(
                style={'width': '100%', 'height': '500px'},
                center=[30.75, -97.48],
                zoom=10,
                children=[dl.TileLayer(id="base-layer-id")]
            )
        ]

    dff = pd.DataFrame.from_dict(viewData)

    # Determine which row to pin; default to first row
    row = 0 if (index is None or len(index) == 0) else index[0]

    # Safely extract coordinates and animal info using column names
    try:
        lat = float(dff.iloc[row]['location_lat'])
        lon = float(dff.iloc[row]['location_long'])
        breed_label = str(dff.iloc[row]['breed'])
        animal_name = str(dff.iloc[row]['name'])
        if not animal_name or animal_name.lower() in ('nan', 'none', ''):
            animal_name = "Unnamed"
    except (IndexError, KeyError, ValueError, TypeError):
        # Fallback to Austin city center
        lat, lon = 30.75, -97.48
        breed_label = "Unknown breed"
        animal_name = "Unknown"

    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[lat, lon],
            zoom=12,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(breed_label),          # Hover: shows breed
                        dl.Popup([                        # Click: shows name
                            html.H4("Animal Name"),
                            html.P(animal_name)
                        ])
                    ]
                )
            ]
        )
    ]


# Run the Dash application in JupyterLab / Codio environment
app.run_server()

Successfully connected to MongoDB
Dash app running on https://oxfordcredit-lifebernard-3000.codio.io/proxy/8050/
